#### BM25 Retrieval Demo for RAG

This notebook demonstrates how to use BM25 as a retriever in a Retrieval-Augmented Generation (RAG) pipeline, using a large Wikipedia dataset.

**1. Install and Import Required Libraries**

Install `rank_bm25`, `nltk`, and `pandas` if not already installed.

In [12]:
# If running for the first time, uncomment the following lines:

# !pip install pandas rank_bm25 nltk

import pandas as pd

from rank_bm25 import BM25Okapi

from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bhupe\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

**2. Load the Wikipedia Dataset**

Load the CSV file with Wikipedia articles. For demo purposes, you may want to sample a subset if the file is very large.

In [13]:
pd.set_option('display.max_colwidth', None)  # Show full text in DataFrame outputs

In [14]:
# https://docs.weaviate.io/weaviate/tutorials/wikipedia

csv_location = r'D:\AI-DATASETS\02-MISC-large\GenAI-LLMs\chromadb\data\vector_database_wikipedia_articles_embedded.csv'

# Only load the columns needed for BM25
df = pd.read_csv(csv_location, usecols=['id', 'title', 'text'])

In [16]:
# Optional: sample for speed in demo
# df = df.sample(10000, random_state=42)
df.sample(5)

,id,title,text
3853,11555,The Kingston Trio,"The Kingston Trio was a folk music group from Palo Alto, California who were popular in the 1950s and 1960s.\n\nReferences\n\nMusical groups from California\nFolk music groups\nPalo Alto, California"
9642,32879,Lelystad,"Lelystad is a town in the middle of the Netherlands.\nIt has about 71,000 inhabitants. It is also the capital of Flevoland, one of the provinces of the Netherlands.\n\nOther websites \n Lelystad website\n\nSettlements in Flevoland\nMunicipalities of Flevoland\nProvincial capitals of the Netherlands"
9896,33879,Detmold,"Detmold (; West Low German: Deppelt) is a town in the German state North Rhine-Westphalia. It has about 74,000 inhabitants. Detmold is the cultural capital of the District of Lippe.\n\nOther websites\nGoogle Maps\n\nReferences\n\nOther websites\n\nLippe Rural District"
22354,84701,Cairnryan,"Cairnryan is a small Scottish village overlooking Loch Ryan and is notable today for its large modern ferry port, operated by P&O, which links Scotland with Larne in Northern Ireland. The village has been of vital importance in maritime history.\n\nOther websites\n Larne Ferry Web - for news & history of ferries to Larne\n\nVillages in Scotland"
23588,90918,Pizza Hut,"Pizza Hut is an American pizza restaurant, or pizza parlor. Pizza Hut also serves salads, pastas and bread sticks. In 2008, Pizza Hut serves chicken wings as a part of the Wingstreet restaurant franchise logo. Pizza Hut is an American restaurant chain and international franchise founded in 1958 by Dan and Frank Carney. The company is known for its Italian-American cuisine menu, including pizza and pasta, as well as side dishes and desserts.\n\nHistory \nPizza Hut was founded in 1958 in Wichita, Kansas, by Dan and Frank Carvey. The first Pizza Hut location was at a intersection in Wichita.\n\nReferences\n\nOther websites\nOfficial website\nPiDubai website \nOfficial Pizza Hut Indonesia website\n\n1958 establishments in the United States\nAmerican fast food restaurants\nCompanies based in Texas\nPlano, Texas\n20th-century establishments in Kansas"


**3. Build the BM25 Index**

Tokenize the text and build a BM25 index for fast retrieval.

In [8]:
# take abt 2 mins
corpus = df['text'].astype(str).tolist()

tokenized_corpus = [word_tokenize(doc.lower()) for doc in corpus]

bm25             = BM25Okapi(tokenized_corpus)

**What does the `bm25` object store?**

- **Tokenized Corpus**: List of tokenized documents (each document is a list of tokens/words).
- **Document Frequencies**: How many documents each term appears in (for IDF calculation).
- **Inverse Document Frequencies (IDF)**: The IDF value for each term in the vocabulary.
- **Average Document Length**: The mean length of all documents in the corpus.
- **Document Lengths**: The length (number of tokens) of each document.
- **Term Frequencies**: For each document, how many times each term appears.

It does **not** store the original text, only the tokenized version and statistics needed for BM25 scoring. You still need your original DataFrame to map results back to the full text.

**4. Retrieve Top-k Documents for a Query**

Define a function to retrieve the most relevant documents for a user query using BM25.

In [17]:
def bm25_retrieve(query, top_k=5):
    # Converts the query to lowercase and splits it into tokens (words),
    # removes stopwords and punctuation for BM25 scoring.
    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords
    import string
    
    stop_words      = set(stopwords.words('english'))
    punct           = set(string.punctuation)
    
    tokenized_query = [t for t in word_tokenize(query.lower()) if t not in stop_words and t not in punct]

    # Calculates a BM25 relevance score for the query against every document in the corpus.
    scores      = bm25.get_scores(tokenized_query)

    # Finds the indices of the top k documents with the highest BM25 scores.
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

    # Selects the rows from your DataFrame corresponding to the top results,
    # keeping only the id, title, and text columns.
    results     = df.iloc[top_indices][['id', 'title', 'text']].copy()

    # Adds a new column to the results DataFrame containing the BM25 scores for each of the top documents.
    results['score'] = [scores[i] for i in top_indices]

    return results

**5. Example: Retrieve and Display Results**

Run a sample query and display the top results.

In [18]:
# Highlight matching tokens in the text column, but ignore stopwords and punctuation
import re
from nltk.corpus import stopwords
import nltk
from nltk.tokenize import word_tokenize
from IPython.display import display, HTML
import string

# Download stopwords if not already present
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bhupe\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [19]:
query = "What is the history of Beirut?"

results = bm25_retrieve(query, top_k=5)

# Clean up newlines in the text column for better display
results['text'] = results['text'].str.replace('\n', ' ', regex=False)

# Tokenize the query for highlighting, remove stopwords and punctuation
stop_words   = set(stopwords.words('english'))
punct        = set(string.punctuation)

query_tokens = set([t for t in word_tokenize(query.lower()) if t not in stop_words and t not in punct])

# Display the query string without stopwords and punctuation (as markdown for clarity)
query_no_stop = ' '.join([t for t in word_tokenize(query) if t.lower() not in stop_words and t not in punct])
display(HTML(f'<b>Query without stopwords:</b> {query_no_stop}'))

def highlight_text(text):
    def replacer(match):
        return f'<span style="background-color: #ffff99;">{match.group(0)}</span>'
    if not query_tokens:
        return text
    # Build regex pattern for all query tokens (word boundaries, ignore case)
    pattern = r'(' + '|'.join(re.escape(token) for token in query_tokens) + r')'
    return re.sub(pattern, replacer, text, flags=re.IGNORECASE)

results['highlighted_text'] = results['text'].apply(highlight_text)

# Only display rows where at least one query token is present in the text (tokenized, not substring)
def has_token_match(text):
    text_tokens = set([t for t in word_tokenize(text.lower()) if t not in stop_words and t not in punct])
    return not query_tokens.isdisjoint(text_tokens)

filtered_results = results[results['text'].apply(has_token_match)]

# Display as HTML so highlights are visible
if not filtered_results.empty:
    display(HTML(filtered_results[['title', 'score', 'highlighted_text']].to_html(escape=False)))
else:
    display(HTML('<b>No results contain the query keywords.</b>'))

,title,score,highlighted_text
16254,Maghen Abraham Synagogue,14.056139,"The Maghen Abraham Synagogue is the oldest synagogue in Beirut, the capital of Lebanon. History This synagogue was very important to Jewish Lebanese in the early twentieth century. It was built in 1925. It was then named after the son of Abraham Sason. It was used as a place to stay for illegal travelers. Some of the people who traveled without legal papers stayed in the synagogue while going to Palestine. Today, it is now called Israel. In 1976, a year after the civil war began, Joseph Farhi took the Torah scrolls from the synagogue to Geneva. Most of them were sent to Sephardic synagogues in Israel. Israel attacked its enemies in Lebanon. This brought anger by other people towards Lebanese Jews. Lebanese Jews became targets to Islamic militant groups since 1984. But even during the fighting, Yasser Arafat's PLO forces and the Christian Phalangists did protect Wadi Abu Jamil during the 1982 Lebanon War. But, the presence of Palestine Liberation Organization forces in the area brought Israeli attacks that damaged the synagogue itself. The late former Prime Minister Rafik Hariri wanted to rebuild the synagoge, but that never happened. The Talmudic school next to it was broken down. This was to keep view of the beach nearby. Related pages Deir el Qamar Synagogue (Mount Lebanon) Beth Elamen Cemetery History of the Jews in Lebanon Wadi Abu Jamil Notes and references Sources Karam, Dima (October 2003), Beirut Synagogue is a Reminder of a Departed People , Daily Star. Hendler, Sefi (August 19 2006), Beirut’s last Jews, Ynetnews. Synagogues Beirut Buildings and structures in Lebanon 1925 establishments in Asia 20th-century establishments in Lebanon"
11538,American University of Beirut,13.288175,"The American University of Beirut (AUB; ) is the first American university to be built in Beirut, Lebanon. Its old name was the Syrian Protestant College, and it was built in the year 1866. The name was changed to American University of Beirut on November 18, 1920. References Colleges and universities in Asia Beirut 1866 establishments"
16222,Walid Eido,13.182384,"Walid Eido (in Arabic: وليد عيدو) (Beirut, 1942 - Beirut, June 13, 2007) was a Lebanese man who worked for the Lebanese Parliament, a government building in Beirut. He died with his son on June 13, 2007, with eight other people, when a bomb blew up just outside a famous amusement park. This was at the waterside of north Beirut. His early life Walid Eido was Sunni Muslim. He was born in the Bachoura area of Beirut. He finished his studies at the Lebanese University in 1966. In the late 1990s, he worked for the Lebanese law of north Lebanon, but, in the year 2000, he quit to go into political life with Mr. Rafic Hariri and to be in the Lebanese parliament. Later on, he worked for the Lebanese Parliament. He used to belong to Al-Murabitun militant group at the time of the Lebanese civil war, he left the Murabitun when this group starts to expand and take bad members in. Personal life Eido was married with three sons named Khaled, Zaher and Mazen. Eido was a good swimmer and the bomb exploded outside his favorite Beirut beach resort, Sporting Club. 1942 births 2007 deaths Assassinated people Lebanese Muslims Lebanese politicians People from Beirut"
3103,Beirut,13.043064,"Beirut is the capital of Lebanon. It is one of the oldest continuously inhabited cities of the world. It is on a hilly promontory on the eastern Mediterranean surrounded to the east by the snow-capped mountains of Lebanon. Before the civil war it was a cultural center of the Arab World, a major international financial, banking and media center and was called the Switzerland of the Middle East. It was called the Paris of the Middle East. Gallery References Linda Jones Hall, Roman Berytus: Beirut in Late Antiquity, 2004. Samir Kassir, Histoire de Beyrouth, Fayard 2003. Richard Talbert, Barrington Atlas of the Greek and Roman World, (), p. 69. Other we

#### Understanding the BM25 Score
- **BM25 score** is a relevance score that measures how well a document matches the query, based on the presence and frequency of query terms in the document, adjusted for document length and term rarity.
- **Higher score = more relevant**: Documents with higher BM25 scores are considered more relevant to the query.
- **Score is not a probability**: The score is not a probability or percentage; it is a relative value. You can only compare scores within the same retrieval run.
- **Typical range**: Scores are usually positive numbers, often between 0 and 20, but can be higher for very strong matches.
- **Interpretation**: A score of 0 means none of the query tokens appear in the document. The more query tokens (and the more rare they are), the higher the score.

In summary: Use the BM25 score to rank documents—the higher the score, the more relevant the document is to the query. The absolute value is less important than the ranking order.

---

#### When to Use BM25 in RAG or Agentic AI Applications
- **Text-Only or Sparse Retrieval**: BM25 excels when you need to retrieve documents based on exact or near-exact keyword matches, especially in traditional text search, FAQ, or knowledge base scenarios.
- **No Embeddings Available**: Use BM25 when you don't have precomputed embeddings or want a fast, lightweight retriever that doesn't require GPU or large models.
- **High Precision for Specific Queries**: BM25 is strong when users expect results containing specific terms (e.g., legal, medical, or technical search where terminology matters).
- **Hybrid Retrieval**: Combine BM25 with dense (embedding-based) retrievers for best results—BM25 can catch keyword matches that embeddings might miss, and vice versa.
- **Small/Medium Datasets**: BM25 is efficient and effective for small to medium-sized corpora. For very large datasets, dense retrieval may scale better, but BM25 is still useful for initial filtering.
- **Transparency and Debugging**: BM25 scores are interpretable and easy to debug, making it a good choice for applications where explainability is important.
- **Agentic AI/Tool Use**: In agentic or tool-using AI, BM25 can be used for fast lookup of tool documentation, code snippets, or structured knowledge where keyword presence is critical.

**Summary:** 
- Use BM25 when you want fast, interpretable, and precise keyword-based retrieval, either alone or as part of a hybrid RAG pipeline. 
- For semantic or fuzzy matching, combine it with embedding-based methods.

---
#### Practical Dos and Don'ts for BM25 (and Preprocessing)
**Dos:**
- **Lowercase all text and queries** before tokenization to ensure case-insensitive matching.
- **Remove punctuation** from both documents and queries for consistent tokenization.
- **Remove stopwords** if you want to focus on meaningful keywords (as in this notebook).
- **Tokenize consistently**: Use the same tokenizer for both corpus and queries.
- **Keep original text**: Store the original (untokenized) text for display and context.
- **Tune BM25 parameters** (`k1`, `b`) if you want to optimize retrieval for your dataset.
- **Combine with dense retrieval** for best results in hybrid RAG systems.
- **Check for empty queries**: Handle cases where all query tokens are removed as stopwords/punctuation.

**Don'ts:**
- **Don't mix tokenization methods** between corpus and queries (e.g., don't use whitespace for one and NLTK for the other).
- **Don't use BM25 for semantic similarity**—it is designed for exact or near-exact keyword matches, not meaning.
- **Don't ignore preprocessing**: Inconsistent preprocessing leads to poor retrieval quality.
- **Don't expect BM25 to handle typos, synonyms, or fuzzy matches**—use query expansion or hybrid methods for that.
- **Don't use BM25 alone for very large or highly diverse datasets**—consider hybrid or dense retrieval for better recall.

**Summary:**
Consistent, clean preprocessing and tokenization are critical for BM25 performance. Use BM25 for fast, interpretable keyword-based retrieval, and combine with other methods for more semantic or robust search.

---
#### Should BM25 or Dense Retrieval Come First in Hybrid RAG?
- **BM25 First (Sparse → Dense):**
    - Use BM25 to quickly filter a large corpus to a smaller candidate set (e.g., top 100–1000 docs), then rerank or further filter with dense retrieval (embeddings).
    - Efficient for large datasets and ensures all candidates contain at least some query keywords.
    - Good when keyword presence is important or you want to avoid semantic drift.

- **Dense First (Dense → Sparse):**
    - Use dense retrieval to get semantically similar candidates, then rerank or filter with BM25 to ensure keyword/term presence.
    - Useful if you want to prioritize semantic similarity but still want to ensure some keyword match.

**Best Practice:**
- In most RAG and hybrid systems, BM25 is often used first for speed and keyword filtering, followed by dense retrieval for semantic reranking.
- For small datasets, you can try both orders and compare results.

**Summary:**
- For large corpora: BM25 first, then dense.
- For semantic focus: Dense first, then BM25.
- For best results: Experiment and combine both (hybrid reranking).